# BigQuery Anti-Pattern Recognition - Setup and Deployment

This notebook will guide you through setting up and deploying the BigQuery Anti-Pattern Recognition tool to Google Cloud Run.

## What this notebook does:
1. Install required dependencies
2. Configure Google Cloud project settings
3. Build the anti-pattern recognition container
4. Deploy to Cloud Run
5. Set up BigQuery connections for UDF usage
6. Save configuration for subsequent notebooks

## Prerequisites:
- Google Cloud Project with billing enabled
- `gcloud` CLI installed and authenticated
- Required APIs enabled (we'll check and enable them)

---

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -r requirements.txt -q

print("✅ Dependencies installed successfully!")

In [ ]:
# Import required libraries
import os
import sys
import json
import time
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Import our utilities
from utils import ConfigManager, DeploymentHelper, display_progress

print("✅ Libraries imported successfully!")

## Step 2: Configuration Setup

Let's configure your Google Cloud project settings:

In [ ]:
# Initialize configuration manager
config = ConfigManager()

# Create configuration widgets
project_id_widget = widgets.Text(
    value=config.get('project_id', ''),
    description='Project ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

region_widget = widgets.Dropdown(
    options=['us-central1', 'us-east1', 'us-west1', 'europe-west1', 'asia-southeast1'],
    value=config.get('region', 'us-central1'),
    description='Region:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

service_name_widget = widgets.Text(
    value=config.get('service_name', 'antipattern-service'),
    description='Service Name:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

artifact_registry_widget = widgets.Text(
    value=config.get('artifact_registry', 'antipattern-registry'),
    description='Artifact Registry:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

bq_dataset_widget = widgets.Text(
    value=config.get('bq_dataset', 'antipattern_demo'),
    description='BQ Dataset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Display configuration form
print("📝 Configure your deployment settings:")
display(widgets.VBox([
    widgets.HTML("<h3>Deployment Configuration</h3>"),
    project_id_widget,
    region_widget,
    service_name_widget,
    artifact_registry_widget,
    bq_dataset_widget
]))

In [ ]:
# Save configuration
config.update({
    'project_id': project_id_widget.value,
    'region': region_widget.value,
    'service_name': service_name_widget.value,
    'artifact_registry': artifact_registry_widget.value,
    'bq_dataset': bq_dataset_widget.value,
    'deployment_timestamp': datetime.now().isoformat()
})

print(f"✅ Configuration saved!")
print(f"Project ID: {config.get('project_id')}")
print(f"Region: {config.get('region')}")
print(f"Service Name: {config.get('service_name')}")

## Step 3: Prerequisites Check

Let's verify that your environment is ready for deployment:

In [ ]:
# Check gcloud authentication
print("🔐 Checking gcloud authentication...")
if DeploymentHelper.check_gcloud_auth():
    print("✅ gcloud is authenticated")
else:
    print("❌ gcloud is not authenticated")
    print("Please run: gcloud auth login")
    print("And: gcloud auth application-default login")

In [ ]:
# Check required APIs
required_apis = [
    'cloudbuild.googleapis.com',
    'run.googleapis.com',
    'artifactregistry.googleapis.com',
    'bigquery.googleapis.com',
    'bigqueryconnection.googleapis.com'
]

print("🔍 Checking required APIs...")
api_status = DeploymentHelper.check_apis_enabled(config.get('project_id'), required_apis)

all_enabled = True
for api, enabled in api_status.items():
    status = "✅" if enabled else "❌"
    print(f"{status} {api}: {'Enabled' if enabled else 'Disabled'}")
    if not enabled:
        all_enabled = False

if not all_enabled:
    print("\n⚠️  Some APIs are not enabled. Enable them with:")
    print(f"gcloud services enable {' '.join(required_apis)} --project={config.get('project_id')}")
else:
    print("\n✅ All required APIs are enabled!")

## Step 4: Enable APIs (if needed)

Run this cell if any APIs need to be enabled:

In [ ]:
# Enable APIs if needed
enable_apis = widgets.Button(
    description='Enable Required APIs',
    button_style='warning',
    layout=widgets.Layout(width='200px')
)

output_widget = widgets.Output()

def on_enable_apis(b):
    with output_widget:
        clear_output()
        print("🔄 Enabling APIs...")
        
        command = f"gcloud services enable {' '.join(required_apis)} --project={config.get('project_id')}"
        success, output = DeploymentHelper.run_command(command)
        
        if success:
            print("✅ APIs enabled successfully!")
        else:
            print(f"❌ Failed to enable APIs: {output}")

enable_apis.on_click(on_enable_apis)

display(widgets.VBox([enable_apis, output_widget]))

## Step 5: Create Artifact Registry Repository

We need a place to store our container image:

In [ ]:
# Create Artifact Registry repository
print("📦 Creating Artifact Registry repository...")

command = f"""
gcloud artifacts repositories create {config.get('artifact_registry')} \
    --repository-format=docker \
    --location={config.get('region')} \
    --description="BigQuery Anti-Pattern Recognition" \
    --project={config.get('project_id')}
"""

success, output = DeploymentHelper.run_command(command)

if success or "already exists" in output:
    print("✅ Artifact Registry repository ready!")
    config.set('artifact_registry_url', 
               f"{config.get('region')}-docker.pkg.dev/{config.get('project_id')}/{config.get('artifact_registry')}")
else:
    print(f"❌ Failed to create repository: {output}")

## Step 6: Build and Deploy Container

Now let's build the anti-pattern recognition service and deploy it to Cloud Run:

In [ ]:
# Build container using Cloud Build
print("🔨 Building container with Cloud Build...")
print("This may take 5-10 minutes...")

container_image = f"{config.get('artifact_registry_url')}/{config.get('service_name')}:latest"

# Change to parent directory for build context
build_command = f"""
cd .. && \
gcloud builds submit . \
    --project={config.get('project_id')} \
    --config=cloudbuild.yaml \
    --substitutions=_CONTAINER_IMAGE_NAME={container_image} \
    --machine-type=e2-highcpu-8
"""

success, output = DeploymentHelper.run_command(build_command)

if success:
    print("✅ Container built successfully!")
    config.set('container_image', container_image)
else:
    print(f"❌ Build failed: {output}")
    print("\nTroubleshooting tips:")
    print("1. Make sure you're running this from the demo/ directory")
    print("2. Check that cloudbuild.yaml exists in the parent directory")
    print("3. Verify your project has Cloud Build API enabled")

In [ ]:
# Deploy to Cloud Run
print("🚀 Deploying to Cloud Run...")

deploy_command = f"""
gcloud run deploy {config.get('service_name')} \
    --image={config.get('container_image')} \
    --region={config.get('region')} \
    --no-allow-unauthenticated \
    --memory=2Gi \
    --cpu=2 \
    --timeout=300 \
    --project={config.get('project_id')}
"""

success, output = DeploymentHelper.run_command(deploy_command)

if success:
    print("✅ Service deployed successfully!")
    
    # Extract service URL
    url_command = f"""
    gcloud run services describe {config.get('service_name')} \
        --region={config.get('region')} \
        --project={config.get('project_id')} \
        --format="value(status.address.url)"
    """
    
    success, service_url = DeploymentHelper.run_command(url_command)
    if success:
        config.set('service_url', service_url.strip())
        print(f"Service URL: {config.get('service_url')}")
else:
    print(f"❌ Deployment failed: {output}")

## Step 7: Create BigQuery Dataset and Connection

Set up BigQuery components for the UDF functionality:

In [ ]:
# Create BigQuery dataset
print("📊 Creating BigQuery dataset...")

dataset_command = f"""
bq mk --dataset \
    --project_id={config.get('project_id')} \
    --location={config.get('region')} \
    --description="Anti-Pattern Recognition Demo" \
    {config.get('bq_dataset')}
"""

success, output = DeploymentHelper.run_command(dataset_command)

if success or "already exists" in output:
    print("✅ BigQuery dataset ready!")
else:
    print(f"❌ Failed to create dataset: {output}")

In [ ]:
# Create BigQuery connection for Cloud Run
print("🔗 Creating BigQuery connection...")

connection_name = f"ext-{config.get('service_name')}"

connection_command = f"""
bq mk --connection \
    --display_name='Anti-Pattern Recognition Connection' \
    --connection_type=CLOUD_RESOURCE \
    --project_id={config.get('project_id')} \
    --location={config.get('region')} \
    {connection_name}
"""

success, output = DeploymentHelper.run_command(connection_command)

if success or "already exists" in output:
    print("✅ BigQuery connection created!")
    config.set('bq_connection', connection_name)
    
    # Get connection service account
    sa_command = f"""
    bq --project_id={config.get('project_id')} --format=json show \
        --connection {config.get('project_id')}.{config.get('region')}.{connection_name}
    """
    
    success, sa_output = DeploymentHelper.run_command(sa_command)
    if success:
        import json
        connection_info = json.loads(sa_output)
        service_account = connection_info['cloudResource']['serviceAccountId']
        config.set('connection_service_account', service_account)
        print(f"Connection service account: {service_account}")
else:
    print(f"❌ Failed to create connection: {output}")

In [ ]:
# Grant Cloud Run Invoker role to connection service account
print("🔐 Granting permissions...")

if config.get('connection_service_account'):
    permission_command = f"""
    gcloud projects add-iam-policy-binding {config.get('project_id')} \
        --member="serviceAccount:{config.get('connection_service_account')}" \
        --role='roles/run.invoker'
    """
    
    success, output = DeploymentHelper.run_command(permission_command)
    
    if success:
        print("✅ Permissions granted successfully!")
    else:
        print(f"❌ Failed to grant permissions: {output}")
else:
    print("⚠️  Skipping permission grant - service account not found")

## Step 8: Create BigQuery Remote Function

Create the UDF that will call our Cloud Run service:

In [ ]:
# Create remote function
print("🔧 Creating BigQuery remote function...")

function_sql = f"""
CREATE OR REPLACE FUNCTION {config.get('bq_dataset')}.get_antipatterns(query STRING)
RETURNS JSON
REMOTE WITH CONNECTION `{config.get('project_id')}.{config.get('region')}.{config.get('bq_connection')}`
OPTIONS (endpoint = '{config.get('service_url')}');
"""

function_command = f"""
bq query --project_id={config.get('project_id')} --use_legacy_sql=false \
    "{function_sql}"
"""

success, output = DeploymentHelper.run_command(function_command)

if success:
    print("✅ Remote function created successfully!")
    config.set('udf_name', f"{config.get('bq_dataset')}.get_antipatterns")
else:
    print(f"❌ Failed to create function: {output}")

## Step 9: Test Deployment

Let's test that everything is working correctly:

In [ ]:
# Test the deployment
print("🧪 Testing deployment...")

# Test Cloud Run service
print("\n1. Testing Cloud Run API...")
try:
    from utils import AntiPatternAnalyzer
    analyzer = AntiPatternAnalyzer(config)
    
    test_query = "SELECT * FROM `bigquery-public-data.samples.shakespeare` LIMIT 10"
    result = analyzer.call_api(test_query)
    
    if 'error' not in result:
        print("✅ Cloud Run API is working!")
        print(f"Found {len(result.get('antipatterns', []))} anti-patterns")
    else:
        print(f"❌ API test failed: {result['error']}")
        
except Exception as e:
    print(f"❌ API test failed: {e}")

# Test BigQuery UDF
print("\n2. Testing BigQuery UDF...")
try:
    test_udf_sql = f"""
    SELECT {config.get('udf_name')}('SELECT * FROM dataset.table') as result
    """
    
    udf_command = f"""
    bq query --project_id={config.get('project_id')} --use_legacy_sql=false \
        --dry_run "{test_udf_sql}"
    """
    
    success, output = DeploymentHelper.run_command(udf_command)
    
    if success:
        print("✅ BigQuery UDF is accessible!")
    else:
        print(f"❌ UDF test failed: {output}")
        
except Exception as e:
    print(f"❌ UDF test failed: {e}")

## Step 10: Deployment Summary

Here's a summary of what was deployed:

In [ ]:
# Display deployment summary
print("🎉 Deployment Summary")
print("=" * 50)

summary_html = f"""
<div style="background-color: #f0f8ff; padding: 20px; border-radius: 10px; margin: 10px 0;">
    <h3>✅ Deployment Completed Successfully!</h3>
    
    <h4>📋 Resources Created:</h4>
    <ul>
        <li><strong>Project:</strong> {config.get('project_id')}</li>
        <li><strong>Region:</strong> {config.get('region')}</li>
        <li><strong>Cloud Run Service:</strong> {config.get('service_name')}</li>
        <li><strong>Service URL:</strong> <a href="{config.get('service_url')}" target="_blank">{config.get('service_url')}</a></li>
        <li><strong>Artifact Registry:</strong> {config.get('artifact_registry')}</li>
        <li><strong>BigQuery Dataset:</strong> {config.get('bq_dataset')}</li>
        <li><strong>BigQuery UDF:</strong> {config.get('udf_name')}</li>
    </ul>
    
    <h4>🚀 Next Steps:</h4>
    <ol>
        <li>Run <strong>02_cloud_run_api_demo.ipynb</strong> to test the API interface</li>
        <li>Run <strong>03_bigquery_udf_demo.ipynb</strong> to test the UDF interface</li>
        <li>Run <strong>04_streamlit_frontend.ipynb</strong> to deploy the web interface</li>
    </ol>
    
    <h4>💡 Quick Test:</h4>
    <p>You can test the UDF directly in BigQuery console:</p>
    <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
SELECT {config.get('udf_name')}('SELECT * FROM dataset.table ORDER BY 1') as antipatterns
    </pre>
</div>
"""

display(HTML(summary_html))

# Save final configuration
config.set('deployment_status', 'completed')
config.set('deployment_completed_at', datetime.now().isoformat())

print("\n💾 Configuration saved to config.json")
print("\n🎯 Ready to proceed with the demo notebooks!")

## Troubleshooting

If you encounter issues:

### Common Issues:

1. **Authentication Errors**
   ```bash
   gcloud auth login
   gcloud auth application-default login
   ```

2. **API Not Enabled**
   ```bash
   gcloud services enable cloudbuild.googleapis.com run.googleapis.com \
       artifactregistry.googleapis.com bigquery.googleapis.com \
       bigqueryconnection.googleapis.com --project=YOUR_PROJECT_ID
   ```

3. **Build Failures**
   - Check that you're in the correct directory
   - Verify cloudbuild.yaml exists in parent directory
   - Check Cloud Build logs in the console

4. **Permission Issues**
   - Ensure your account has necessary IAM roles
   - Check that billing is enabled on the project

### Clean Up (if needed)

To remove all resources:
```bash
# Delete Cloud Run service
gcloud run services delete SERVICE_NAME --region=REGION

# Delete Artifact Registry repository
gcloud artifacts repositories delete REPO_NAME --location=REGION

# Delete BigQuery dataset
bq rm -r -d PROJECT_ID:DATASET_NAME
```